# Give a Letta agent a document to remember: Unstructured Transform MCP + archival memory

This notebook turns a document into knowledge that a [Letta](https://docs.letta.com) agent carries with it. We parse the file with the [Unstructured](https://unstructured.io) **Transform MCP server**, load the result into the agent's [archival memory](https://docs.letta.com/guides/core-concepts/memory/archival-memory), and then ask questions. The agent retrieves only the passages it needs for each question, so the document becomes a reliable reference the agent can draw on in any future conversation.

The [Unstructured Transform MCP server](https://docs.unstructured.io/transform/overview) converts PDFs, DOCX, PPTX, spreadsheets, HTML, images, and 60+ other formats into clean markdown. We drive it with an agent using the OpenAI [Responses API](https://platform.openai.com/docs/api-reference/responses), then hand the markdown to Letta.

## 1. Prerequisites

Three API keys:

- **`LETTA_API_KEY`** from [app.letta.com](https://app.letta.com) for the agent and its memory.
- **`UNSTRUCTURED_API_KEY`** from [platform.unstructured.io](https://platform.unstructured.io), the Bearer token for the Transform MCP server.
- **`OPENAI_API_KEY`** from [platform.openai.com](https://platform.openai.com), used to run the parsing agent.

**A note on the Letta model handles.** The `LETTA_MODEL` and `LETTA_EMBEDDING` set below use `openai/...` handles, which run on Letta's hosted models and draw on your Letta credits. If you would rather use your own OpenAI account, add your OpenAI API key on the Letta Platform: it then shows up as a handle under your provider's name (for example `my-openai/gpt-5.6-sol`) and is billed by OpenAI directly. Set `LETTA_MODEL` and `LETTA_EMBEDDING` to whichever handles your workspace can run (check `letta.models.list()`).

In [ ]:
%pip install --upgrade openai letta-client requests

In [ ]:
import os, re, json, time
import getpass
import requests
from openai import OpenAI
from letta_client import Letta

for var in ("LETTA_API_KEY", "UNSTRUCTURED_API_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

TRANSFORM_MCP_URL = "https://mcp.transform.unstructured.io/"
PDF_URL = "https://arxiv.org/pdf/1706.03762"  # 'Attention Is All You Need'

# Model that runs the parsing agent (OpenAI Responses API).
OPENAI_MODEL = "gpt-5-mini"

# Model and embedding handles for the Letta agent (see letta.models.list()). Bare 'openai/...' and
# 'letta/...' handles bill Letta credits; a key added in the Letta portal appears under its own
# provider prefix and bills that key.
LETTA_MODEL = "openai/gpt-4.1"
LETTA_EMBEDDING = "openai/text-embedding-3-small"

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
letta = Letta(api_key=os.environ["LETTA_API_KEY"])

## 2. Parse the document with an agent

The Transform MCP server is a remote MCP server, so we attach it directly to the OpenAI Responses API as an MCP tool. Parsing runs as an asynchronous job, so we also give the model a `wait_seconds` tool it can call between status checks. The agent submits the job with `transform_files`, polls `check_transform_status` until it is `COMPLETED`, and calls `get_transform_results` for the markdown. We then download and return it.

In [ ]:
def transform_mcp_tool():
    """Responses API MCP tool block for the Unstructured Transform server."""
    return {
        "type": "mcp",
        "server_label": "unstructured_transform",
        "server_url": TRANSFORM_MCP_URL,
        "require_approval": "never",
        "headers": {"Authorization": f"Bearer {os.environ['UNSTRUCTURED_API_KEY']}"},
    }

def wait_seconds(seconds):
    """Wait before checking the job status again, so the agent can wait out a slow job."""
    time.sleep(seconds)
    return f"Waited {seconds} seconds."

WAIT_TOOL = {
    "type": "function", "name": "wait_seconds",
    "description": "Wait the given number of seconds before checking the job status again.",
    "parameters": {"type": "object", "properties": {"seconds": {"type": "integer"}},
                   "required": ["seconds"], "additionalProperties": False},
}

MARKDOWN_URL = re.compile(r"https://mcp\.transform\.unstructured\.io/output/\S+?\.md[^\"\\ ]*")

# Cap on the number of client-side tool round-trips (mostly wait_seconds polls). The loop exits as
# soon as the agent stops calling tools; this is just a safety bound so it can never spin forever.
MAX_TOOL_STEPS = 40

def parse_to_markdown(url):
    """Let the model drive Transform and return the parsed markdown."""
    tools = [transform_mcp_tool(), WAIT_TOOL]
    instruction = (
        f"Parse the PDF at {url} using the Unstructured Transform tools. Call transform_files with the "
        "URL, then poll check_transform_status, calling wait_seconds(10) between each check, until the "
        "status is COMPLETED. Then call get_transform_results with output_format 'md' and report the "
        "markdown download URL."
    )
    resp = openai_client.responses.create(model=OPENAI_MODEL, tools=tools, input=instruction)
    seen = json.dumps(resp.model_dump())
    for _ in range(MAX_TOOL_STEPS):
        calls = [o for o in resp.output if o.type == "function_call"]
        if not calls:
            break
        outputs = []
        for call in calls:
            args = json.loads(call.arguments)
            result = wait_seconds(**args) if call.name == "wait_seconds" else "unknown tool"
            outputs.append({"type": "function_call_output", "call_id": call.call_id, "output": result})
        resp = openai_client.responses.create(model=OPENAI_MODEL, tools=tools,
                                              previous_response_id=resp.id, input=outputs)
        seen += "\n" + json.dumps(resp.model_dump())
    urls = MARKDOWN_URL.findall(seen)
    if not urls:
        raise RuntimeError("Transform did not return a markdown URL; check the job status.")
    return requests.get(urls[0], timeout=120).text

markdown = parse_to_markdown(PDF_URL)
print(f"parsed {len(markdown):,} characters of markdown")

## 3. Chunk the markdown

Transform returns clean markdown with the document's headings intact. We split it into passage-sized chunks and keep the section heading each chunk falls under, so the agent can cite sections in its answers. The result is a simple JSON list of chunks.

In [ ]:
def chunk_markdown(text, max_chars=1200):
    """Split markdown into ~max_chars chunks, tagging each with the section heading it falls under."""
    chunks, buffer, size, section = [], [], 0, "Introduction"
    for block in text.split("\n\n"):
        block = block.strip()
        if not block:
            continue
        is_heading = block.splitlines()[0].lstrip().startswith("#")
        # Close the current chunk at a new heading or when it gets long enough.
        if buffer and (is_heading or size + len(block) > max_chars):
            chunks.append({"section": section, "text": "\n\n".join(buffer)})
            buffer, size = [], 0
        if is_heading:
            section = block.splitlines()[0].lstrip("#").strip() or section
        buffer.append(block)
        size += len(block)
    if buffer:
        chunks.append({"section": section, "text": "\n\n".join(buffer)})
    return chunks

chunks = chunk_markdown(markdown)
print(f"{len(chunks)} chunks")

## 4. Load the chunks into the agent's archival memory

We create a Letta agent and attach its `archival_memory_search` tool, which lets it retrieve stored passages. Then we insert each chunk as an archival-memory passage, prefixed with its section heading. Letta embeds each passage on insert, so the agent can find the right ones by meaning. This ingestion is a one-time step; the passages stay with the agent.

In [ ]:
archival_search = next(t for t in letta.tools.list(limit=200) if t.name == "archival_memory_search")

agent = letta.agents.create(
    name="document-analyst",
    model=LETTA_MODEL,
    embedding=LETTA_EMBEDDING,
    tool_ids=[archival_search.id],
    memory_blocks=[
        {
            "label": "persona",
            "value": (
                "I have NOT read the user's document and I have NO prior knowledge of its contents. The "
                "only way I can learn anything about it is to call archival_memory_search, where it is "
                "stored one passage per section, each prefixed with its section heading. Before answering "
                "ANY question I first call archival_memory_search, then answer strictly from the passages "
                "it returns and cite the section. If the search returns nothing relevant I say so. I never "
                "answer from general knowledge, and I never ask the user to provide or paste the document."
            ),
        },
    ],
)
print("agent id:", agent.id)

for i, chunk in enumerate(chunks):
    letta.agents.passages.create(agent_id=agent.id, text=f"[{chunk['section']}] {chunk['text']}")
    if (i + 1) % 10 == 0 or i == len(chunks) - 1:
        print(f"  loaded {i + 1}/{len(chunks)} passages")

## 5. Ask questions

Retrieval is built in: when we send a question, the agent calls `archival_memory_search`, reads the passages that come back, and answers from them. Each call brings back only the relevant passages, so the whole document never has to fit in the prompt.

In [ ]:
def ask(question):
    response = letta.agents.messages.create(
        agent_id=agent.id,
        messages=[{"role": "user", "content": question}],
    )
    for m in response.messages:
        kind = getattr(m, "message_type", None)
        if kind == "tool_call_message":
            call = getattr(m, "tool_call", None)
            print("  [searched archival memory]" if call and call.name == "archival_memory_search"
                  else f"  [tool] {getattr(call, 'name', '?')}")
        elif kind == "assistant_message":
            print("\n" + str(m.content))
    return response

_ = ask("Using the document in your archival memory, how does multi-head attention work? Cite the section.")

In [ ]:
_ = ask("From your archival memory, what positional encoding does the model use? Cite the section.")

## 6. The document stays in the agent's memory

The passages live with the agent, so it can answer from this document in any later session without re-parsing or re-uploading. Here is what it is holding.

In [ ]:
passages = list(letta.agents.passages.list(agent_id=agent.id))
print(f"{len(passages)} passages in archival memory")
print("example:", passages[0].text[:200])

## Best practices & next steps

- **Give the agent a document once.** After ingestion the agent answers from archival memory across sessions, which makes this a good fit for a support agent over a manual, a policy document, or a research paper.
- **Make use of Tags** - Self-explanatory :) 
- **Only relevant passages enter the prompt.** Each question retrieves a few passages by meaning, so token usage stays low and large documents work fine.
- **Transform handles the hard part.** It turns 60+ messy file formats into clean markdown; limits are 50 MB/file, 10 files/request, and 5 concurrent jobs.

### Resources
- [Unstructured](https://unstructured.io) and the [Unstructured Transform docs](https://docs.unstructured.io/transform/overview)
- [Letta docs](https://docs.letta.com) and [archival memory](https://docs.letta.com/guides/core-concepts/memory/archival-memory)